# Calculo y visualizacion de normales en meshes 3D

Este notebook implementa:
- normales de caras (manuales con producto cruz)
- verificacion/correccion de orientacion
- normales de vertices (promedio de caras adyacentes)
- comparacion con `trimesh`
- comparacion Flat vs Smooth shading
- validacion de magnitud y consistencia


In [ ]:
# En Colab puedes ejecutar esta celda para instalar dependencias:
# %pip -q install trimesh numpy matplotlib plotly vedo

import numpy as np
import trimesh
import matplotlib.pyplot as plt
from pathlib import Path
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from matplotlib import cm
import urllib.request
import plotly.graph_objects as go

np.set_printoptions(precision=5, suppress=True)


In [ ]:
# Descarga automatica de un modelo simple y gratuito (Khronos Box.glb).
DEFAULT_MODEL_URL = "https://raw.githubusercontent.com/KhronosGroup/glTF-Sample-Assets/main/Models/Box/glTF-Binary/Box.glb"
MODEL_PATH = Path("Box.glb")

if not MODEL_PATH.exists():
    print(f"Descargando modelo de ejemplo desde: {DEFAULT_MODEL_URL}")
    urllib.request.urlretrieve(DEFAULT_MODEL_URL, str(MODEL_PATH))
    print(f"Modelo guardado en: {MODEL_PATH.resolve()}")
else:
    print(f"Modelo existente encontrado en: {MODEL_PATH.resolve()}")

# Si quieres usar otro archivo propio, cambia esta variable:
# MODEL_PATH = Path("mi_modelo.obj")

# Ayuda rapida: listar archivos 3D en carpeta actual.
candidatos = []
for ext in ("*.obj", "*.stl", "*.gltf", "*.glb", "*.ply"):
    candidatos.extend(Path(".").glob(ext))
print("Modelos detectados:", [str(p) for p in candidatos])


In [ ]:
def load_mesh(path: Path) -> trimesh.Trimesh:
    if not path.exists():
        raise FileNotFoundError(f"No existe el archivo: {path}")

    loaded = trimesh.load(path, force="mesh", process=False)

    # Algunos formatos pueden cargar como Scene.
    if isinstance(loaded, trimesh.Scene):
        if not loaded.geometry:
            raise ValueError("La escena no contiene geometria.")
        mesh = trimesh.util.concatenate(tuple(loaded.geometry.values()))
    else:
        mesh = loaded

    if not isinstance(mesh, trimesh.Trimesh):
        raise TypeError(f"Tipo no soportado: {type(mesh)}")

    mesh = mesh.copy()
    mesh.remove_unreferenced_vertices()
    return mesh


def compute_face_normals_manual(vertices, faces, eps=1e-12):
    tri = vertices[faces]
    v1 = tri[:, 1] - tri[:, 0]
    v2 = tri[:, 2] - tri[:, 0]
    normals = np.cross(v1, v2)
    lengths = np.linalg.norm(normals, axis=1, keepdims=True)

    valid = lengths[:, 0] > eps
    normals_unit = np.zeros_like(normals)
    normals_unit[valid] = normals[valid] / lengths[valid]
    return normals_unit, valid


def face_orientation_dots(face_centers, normals, reference_point):
    direction = face_centers - reference_point
    dots = np.einsum("ij,ij->i", direction, normals)
    return dots


def orient_normals_outward(mesh, face_normals):
    dots = face_orientation_dots(mesh.triangles_center, face_normals, mesh.centroid)
    pos = np.count_nonzero(dots > 0)
    neg = np.count_nonzero(dots < 0)
    flipped_globally = False

    # Si la mayoria apunta hacia adentro, invertir todas.
    if neg > pos:
        face_normals = -face_normals
        dots = -dots
        flipped_globally = True

    return face_normals, dots, flipped_globally


In [ ]:
def compute_face_areas(vertices, faces):
    tri = vertices[faces]
    v1 = tri[:, 1] - tri[:, 0]
    v2 = tri[:, 2] - tri[:, 0]
    return 0.5 * np.linalg.norm(np.cross(v1, v2), axis=1)


def compute_vertex_normals_manual(vertices, faces, face_normals, eps=1e-12):
    areas = compute_face_areas(vertices, faces)
    weighted = face_normals * areas[:, None]

    vnormals = np.zeros((len(vertices), 3), dtype=np.float64)
    np.add.at(vnormals, faces[:, 0], weighted)
    np.add.at(vnormals, faces[:, 1], weighted)
    np.add.at(vnormals, faces[:, 2], weighted)

    lengths = np.linalg.norm(vnormals, axis=1, keepdims=True)
    valid = lengths[:, 0] > eps
    vnormals_unit = np.zeros_like(vnormals)
    vnormals_unit[valid] = vnormals[valid] / lengths[valid]
    return vnormals_unit, valid


def validate_unit_normals(normals, tol=1e-5):
    mags = np.linalg.norm(normals, axis=1)
    ok = np.all(np.abs(mags - 1.0) < tol)
    return ok, mags


def orientation_consistency(dots):
    # Fraccion de caras que apuntan 'hacia afuera' segun el centroide.
    return np.mean(dots > 0)


def correct_mesh_orientation(mesh):
    corrected = mesh.copy()
    corrected.fix_normals()
    return corrected


In [ ]:
def set_axes_equal(ax, points):
    points = np.asarray(points)
    mins = points.min(axis=0)
    maxs = points.max(axis=0)
    center = (mins + maxs) / 2.0
    radius = (maxs - mins).max() / 2.0

    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    ax.set_zlim(center[2] - radius, center[2] + radius)


def plot_normals_interactive(mesh, normals, mode="face", sample=500, length=0.06, title="Normals", cone_size=0.35):
    if mode == "face":
        origins = mesh.triangles_center
    elif mode == "vertex":
        origins = mesh.vertices
    else:
        raise ValueError("mode debe ser 'face' o 'vertex'")

    if len(origins) > sample:
        idx = np.linspace(0, len(origins) - 1, sample, dtype=int)
        origins = origins[idx]
        dirs = normals[idx]
    else:
        dirs = normals

    vectors = dirs * length

    mesh_trace = go.Mesh3d(
        x=mesh.vertices[:, 0], y=mesh.vertices[:, 1], z=mesh.vertices[:, 2],
        i=mesh.faces[:, 0], j=mesh.faces[:, 1], k=mesh.faces[:, 2],
        color="lightgray", opacity=0.35, name="Mesh", showscale=False
    )

    normal_arrows = go.Cone(
        x=origins[:, 0], y=origins[:, 1], z=origins[:, 2],
        u=vectors[:, 0], v=vectors[:, 1], w=vectors[:, 2],
        anchor="tail",
        sizemode="absolute",
        sizeref=cone_size * length,
        colorscale=[[0.0, "red"], [1.0, "red"]],
        showscale=False,
        name=f"Normals ({mode})",
        hoverinfo="skip"
    )

    fig = go.Figure(data=[mesh_trace, normal_arrows])
    fig.update_layout(
        title=title,
        height=760,
        scene=dict(
            aspectmode="data",
            xaxis=dict(title="X"),
            yaxis=dict(title="Y"),
            zaxis=dict(title="Z"),
            dragmode="orbit"
        ),
        margin=dict(l=0, r=0, b=0, t=40),
        legend=dict(x=0.01, y=0.99)
    )
    fig.show(config={"scrollZoom": True, "displaylogo": False})


In [ ]:
mesh = load_mesh(MODEL_PATH)
print(mesh)

# Normales de caras manuales.
face_normals_manual, valid_faces = compute_face_normals_manual(mesh.vertices, mesh.faces)
face_normals_manual, dots, flipped = orient_normals_outward(mesh, face_normals_manual)

# Normales de caras de trimesh.
face_normals_tm = mesh.face_normals.copy()

# Alinear signo para comparacion numerica estable.
sign = np.sign(np.einsum("ij,ij->i", face_normals_manual, face_normals_tm))
sign[sign == 0] = 1.0
face_normals_tm_aligned = face_normals_tm * sign[:, None]

face_diff = np.linalg.norm(face_normals_manual - face_normals_tm_aligned, axis=1)

# Normales de vertices manuales.
vertex_normals_manual, valid_vertices = compute_vertex_normals_manual(
    mesh.vertices, mesh.faces, face_normals_manual
)
vertex_normals_tm = mesh.vertex_normals.copy()

sign_v = np.sign(np.einsum("ij,ij->i", vertex_normals_manual, vertex_normals_tm))
sign_v[sign_v == 0] = 1.0
vertex_normals_tm_aligned = vertex_normals_tm * sign_v[:, None]

vertex_diff = np.linalg.norm(vertex_normals_manual - vertex_normals_tm_aligned, axis=1)

print(f"Caras validas: {valid_faces.sum()} / {len(valid_faces)}")
print(f"Vertices validos: {valid_vertices.sum()} / {len(valid_vertices)}")
print(f"Se invirtieron globalmente normales de cara?: {flipped}")
print(f"Diferencia media face normal manual vs trimesh: {face_diff.mean():.6e}")
print(f"Diferencia media vertex normal manual vs trimesh: {vertex_diff.mean():.6e}")


In [ ]:
# Visualizacion 3D interactiva: rota con click izquierdo, panea con derecho, zoom con rueda.
diag = np.linalg.norm(mesh.bounds[1] - mesh.bounds[0])
arrow_len = 0.08 * diag

plot_normals_interactive(
    mesh, face_normals_manual, mode="face", sample=300,
    length=arrow_len, title="Normales de caras (flechas interactivas)", cone_size=0.42
)

plot_normals_interactive(
    mesh, vertex_normals_manual, mode="vertex", sample=500,
    length=arrow_len, title="Normales de vertices (flechas interactivas)", cone_size=0.32
)


In [ ]:
def render_shading_comparison(mesh, face_normals, vertex_normals):
    light_dir = np.array([0.35, 0.45, 1.0], dtype=float)
    light_dir /= np.linalg.norm(light_dir)

    # Flat: intensidad por cara.
    i_face = np.clip(np.einsum("ij,j->i", face_normals, light_dir), 0, 1)

    # Smooth (aprox): intensidad por vertice y promedio por triangulo.
    i_vertex = np.clip(np.einsum("ij,j->i", vertex_normals, light_dir), 0, 1)
    i_smooth_face = i_vertex[mesh.faces].mean(axis=1)

    tris = mesh.vertices[mesh.faces]

    fig = plt.figure(figsize=(14, 6))
    ax1 = fig.add_subplot(121, projection="3d")
    ax2 = fig.add_subplot(122, projection="3d")

    poly_flat = Poly3DCollection(tris, linewidths=0.05, edgecolors="black")
    poly_flat.set_facecolor(cm.viridis(i_face))
    ax1.add_collection3d(poly_flat)
    ax1.set_title("Flat shading (normal por cara)")

    poly_smooth = Poly3DCollection(tris, linewidths=0.05, edgecolors="black")
    poly_smooth.set_facecolor(cm.viridis(i_smooth_face))
    ax2.add_collection3d(poly_smooth)
    ax2.set_title("Smooth shading aprox (promedio de normales de vertice)")

    for ax in (ax1, ax2):
        set_axes_equal(ax, mesh.vertices)
        ax.set_axis_off()

    plt.tight_layout()
    plt.show()


render_shading_comparison(mesh, face_normals_manual, vertex_normals_manual)


In [ ]:
# Opcional: comparacion interactiva Flat vs Smooth con vedo (si esta instalado).
try:
    from vedo import Mesh, Plotter

    vmesh_data = [mesh.vertices, mesh.faces]
    actor_flat = Mesh(vmesh_data).c("lightgray").flat()
    actor_smooth = Mesh(vmesh_data).c("lightgray").phong()

    pltv = Plotter(N=2, axes=1, sharecam=False, bg="white")
    pltv.at(0).show(actor_flat, "Flat (vedo)", resetcam=True)
    pltv.at(1).show(actor_smooth, "Smooth (vedo)", resetcam=True)
    pltv.interactive().close()
except Exception as e:
    print("Vedo no disponible o fallo al renderizar:", e)


In [ ]:
face_ok, face_mags = validate_unit_normals(face_normals_manual)
vert_ok, vert_mags = validate_unit_normals(vertex_normals_manual)

inverted_idx = np.where(dots < 0)[0]
consistency = orientation_consistency(dots)

print("=== Validacion ===")
print(f"Magnitud unitaria en normales de cara?: {face_ok}")
print(f"Magnitud unitaria en normales de vertice?: {vert_ok}")
print(f"Normales de cara invertidas detectadas: {len(inverted_idx)}")
print(f"Consistencia de orientacion (0-1): {consistency:.4f}")

if len(inverted_idx) > 0:
    print("Se recomienda corregir orientacion del mesh con trimesh.fix_normals().")
else:
    print("No se detectaron caras invertidas segun el criterio centroide->cara.")


In [ ]:
# Correccion opcional de orientacion de malla completa.
mesh_corrected = correct_mesh_orientation(mesh)

# Recalculo despues de correccion.
fn_corr, _ = compute_face_normals_manual(mesh_corrected.vertices, mesh_corrected.faces)
fn_corr, dots_corr, _ = orient_normals_outward(mesh_corrected, fn_corr)
cons_corr = orientation_consistency(dots_corr)
print(f"Consistencia luego de fix_normals(): {cons_corr:.4f}")
